# Beyond Words — Phase 6: fine-tune a pretrained encoder for dialect ID (Kaggle)

Runs `finetune_xlsr.py` with the Phase 2 recipe for one encoder and one split protocol.
Set `MODEL`, `TAG`, `FAMILY`, `SPLIT`, `EPOCHS` in the second cell.

**Notebook settings:** GPU T4 (not P100), Internet ON, input dataset `bangla-accent-voice-data`.
Outputs in `/kaggle/working`: `{TAG}_best.pt`, `metrics_{TAG}.json`, `test_predictions_{TAG}.csv`.

In [ ]:
!git clone --depth 1 https://github.com/towfique-elahe/beyond-words.git /tmp/beyond-words
!nvidia-smi -L

In [ ]:
MODEL = 'openai/whisper-medium'   # or bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium,
                                  #    bengaliAI/tugstugi_bengaliai-asr_whisper-medium,
                                  #    hishab/hishab_bn_fastconformer (FAMILY='nemo')
TAG = 'whisper_medium'
FAMILY = 'auto'                   # auto | wav2vec2 | whisper | nemo
SPLIT = 'manifest_split_strict_rel.csv'   # or manifest_split_rel.csv
EPOCHS = 12                       # 20 for the small Hishab encoder
EXTRA_ARGS = ''                   # e.g. '--grad-ckpt' if CUDA runs out of memory

from pathlib import Path
AUDIO_ROOT = None
for p in Path('/kaggle/input').rglob('Barishal'):
    if p.is_dir():
        AUDIO_ROOT = str(p.parent)
        break
assert AUDIO_ROOT, 'Dataset not found — attach bangla-accent-voice-data as input'
print('AUDIO_ROOT =', AUDIO_ROOT)

In [ ]:
import time
if FAMILY == 'nemo':
    !pip install -q "nemo_toolkit[asr]"
t0 = time.time()
!python /tmp/beyond-words/finetune_xlsr.py \
    --model {MODEL} --family {FAMILY} --tag {TAG} \
    --manifest /tmp/beyond-words/splits/{SPLIT} \
    --audio-root {AUDIO_ROOT} --out-dir /kaggle/working \
    --epochs {EPOCHS} --batch 8 --accum 2 --workers 4 {EXTRA_ARGS}
print(f'{TAG} {SPLIT}: {(time.time() - t0) / 60:.1f} min')

In [ ]:
import json
print(json.dumps(json.load(open(f'/kaggle/working/metrics_{TAG}.json')), indent=2))